<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 1 · Un *pipeline* completo, de los datos crudos a la evaluación

**Unidad 1**

En este laboratorio vas a recorrer tú solo el flujo de trabajo que vimos en clase, con datos que no has visto: revisar los datos, separar, preparar cada tipo de columna, comparar contra una línea base y evaluar una sola vez.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "rediseno-2027"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u01-lab01-pipeline-censo"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/01-introduccion/laboratorios/tests/u01-lab01-pipeline-censo/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

## Los datos

Una muestra de 12,000 personas del censo de Estados Unidos de 1994, preparada por Ronny Kohavi y Barry Becker y distribuida por el repositorio UCI con el nombre *Adult*. Para cada persona hay datos de edad, escolaridad, ocupación, estado civil, horas de trabajo a la semana y otros.

**Problema.** Predecir si la persona gana **más de 50,000 dólares al año** (columna `ingreso`).

Es uno de los conjuntos de datos más usados para enseñar clasificación y también para estudiar sesgos: incluye sexo y raza. Volveremos a eso al final.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

import rdp

rdp.estilo.usar()

## 1. Revisar los datos

Cargamos los datos tal cual y revisamos si hay faltantes.

In [ ]:
crudo = rdp.datos.cargar("censo_adult")
crudo.isna().sum().sum()

Ninguno. Pero mira los valores de la columna `workclass`:

In [ ]:
crudo["workclass"].value_counts()

**Pregunta 1.** En estos datos, los faltantes vienen escritos como `"?"`. Vuelve a cargar los datos en la variable `censo` de modo que pandas los reconozca como faltantes. Pista: `rdp.datos.cargar` acepta los mismos argumentos que `pd.read_csv`, entre ellos `na_values`.

In [ ]:
censo = ...

In [ ]:
grader.check("p1")

**Pregunta 2.** Construye la variable objetivo `y` como enteros: 1 si `ingreso` es `">50K"` y 0 si no. En `X` deja todas las demás columnas. Después separa un 20 % para prueba, **estratificando** por `y` y con `random_state=0`, en `X_tr, X_te, y_tr, y_te`.

In [ ]:
y = ...
X = ...
X_tr, X_te, y_tr, y_te = ...

In [ ]:
grader.check("p2")

## 2. Entender las columnas

Antes de decidir cómo tratar cada columna, hay que entender qué significa. Dos columnas merecen atención:

In [ ]:
pd.crosstab(X_tr["education"], X_tr["education-num"]).idxmax(axis=1).sort_values()

`education-num` es exactamente la codificación **ordinal** de `education`: Preschool = 1, …, Doctorate = 16. Usar las dos sería repetir la misma información.

`fnlwgt` (*final weight*) no describe a la persona: es un **peso muestral** del censo, que indica cuántas personas de la población representa ese registro. No es un atributo con el que tenga sentido predecir el ingreso de alguien.

**Pregunta 3.** Define tres listas con nombres de columnas:

- `numericas`: `age`, `education-num` y `hours-per-week`;
- `sesgadas`: `capital-gain` y `capital-loss` (la mayoría de las personas tiene 0 y unas pocas, cantidades enormes);
- `nominales`: las siete columnas categóricas **sin orden**.

No incluyas `education` (ya está en `education-num`) ni `fnlwgt`.

In [ ]:
numericas = ...
sesgadas = ...
nominales = ...

In [ ]:
grader.check("p3")

**Pregunta 4.** Construye un `ColumnTransformer` llamado `preprocesamiento` con tres ramas:

1. `numericas`: imputar con la mediana y estandarizar;
2. `sesgadas`: aplicar `np.log1p` y estandarizar;
3. `nominales`: imputar los faltantes con la constante `"desconocido"` y codificar con *one-hot*, ignorando categorías desconocidas y agrupando las que aparecen menos de 20 veces.

Las columnas que no menciones (`education`, `fnlwgt`) se descartan automáticamente.

In [ ]:
preprocesamiento = ColumnTransformer([
    ("numericas", ..., numericas),
    ("sesgadas", ..., sesgadas),
    ("nominales", ..., nominales),
])

In [ ]:
grader.check("p4")

## 3. Línea base y modelo

Usamos validación cruzada estratificada de 5 particiones **solo con el entrenamiento**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

**Pregunta 5.** Calcula con `cross_val_score` y `cv`:

- `exactitud_base`: la exactitud media de un `DummyClassifier` que siempre predice la clase más frecuente;
- `exactitud_modelo`: la exactitud media de un `Pipeline` llamado `modelo` que encadena `preprocesamiento` y una `LogisticRegression(max_iter=1000)`.

In [ ]:
exactitud_base = ...
modelo = ...
exactitud_modelo = ...
print(f"línea base: {exactitud_base:.3f} · modelo: {exactitud_modelo:.3f}")

In [ ]:
grader.check("p5")

**Pregunta 6.** Ya elegimos el modelo. Entrénalo con todo el conjunto de entrenamiento y guarda en `exactitud_prueba` su exactitud en el conjunto de prueba. Es la **única** vez que usamos la prueba.

In [ ]:
...
exactitud_prueba = ...
exactitud_prueba

In [ ]:
grader.check("p6")

## 4. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una.

In [ ]:
pd.crosstab(X_tr["relationship"], X_tr["sex"])

<!-- BEGIN QUESTION -->

**Reflexión 1.** Supón que, para que el modelo «no discrimine por sexo», quitamos la columna `sex`. Con base en la tabla anterior, ¿el modelo deja de poder usar el sexo de la persona? Explica qué es una **variable sustituta** (*proxy*) y da otro ejemplo de este conjunto de datos.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** Repite la pregunta 5 sin aplicar el logaritmo a `capital-gain` y `capital-loss` (trátalas como las demás numéricas). ¿Mejoró o empeoró la exactitud? ¿Contradice eso lo que vimos en clase sobre las variables sesgadas? Argumenta.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()